# Music Annotation Project V9.0A
## Mask-Authority Transfer Experiment, revised Notebook 06/07 identity handling

**Runtime:** CPU when Notebook 06 confidence maps already exist.

This clean revision fixes the Notebook 06 to Notebook 07 namespace transition:

- Notebook 06 artefacts retain the original annotated-source stem.
- Notebook 07 assigns the canonical downstream `PageXXX` identity.
- Cross-stage matching is performed only with `source_md5`.
- V9.0A uses the Notebook 07 `page` field as its canonical page identity.
- The Notebook 06 confidence map is resolved from the matching draft audit. It is never guessed as `PageXXX_confidence.npy`.

The experiment compares these mask-authority arms on the same resolved pages:

1. `reviewed_comparator`: Notebook 07 corrected mask.
2. `raw_yolo`: Notebook 06 draft mask.
3. `confidence_gated_yolo`: Notebook 06 confidence map thresholded with the previously established YOLO gate.
4. `reviewer_corrected_yolo`: the Notebook 07 corrected mask paired to that exact Notebook 06 draft lineage.

The reviewed comparator and reviewer-corrected YOLO arms are intentionally pixel-identical in the present HITL lineage. The former names the experimental comparator role; the latter names the operational provenance role. The notebook records this rather than pretending they are distinct masks.

### Fail-closed controls

- No filename-stem join between Notebooks 06 and 07.
- No use of the frozen V7.8 independent set as ordinary V9.0A development data.
- No clean page is loaded before mask and correction outputs are locked.
- No detector threshold is optimised here.
- No automatic mask authority or production output is authorised.

In [ ]:
# PROVENANCE: standard project Colab bootstrap.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
# PROVENANCE: project paths from Notebooks 06, 07 and 09; new V9.0A output layout.
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, sys, platform, re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
CLEAN_DIR=ROOT/'source'/'clean'
OUTPUT_ROOT=ROOT/'mask_authority_transfer_v9_0a'

# Canonical PageXXX allowlist. None means every complete Notebook 06/07 pair.
PAGE_ALLOWLIST=None

# Previously established YOLO confidence-gating policy.
# Set this to the exact established detector-confidence threshold before execution.
# It is deliberately not inferred from Notebook 06 `prediction_confidence`, which defines the raw draft mask.
ESTABLISHED_YOLO_GATE=None
ESTABLISHED_YOLO_GATE_POLICY_ID=None

# Frozen correction contract carried by the canonical V7.8/Notebook 09 implementation.
MASK_DILATION=2
CROP_MARGIN=48
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0a_mask_authority_transfer'
OUT=OUTPUT_ROOT/'runs'/RUN_ID
INPUTS=OUT/'01_inputs'; MASKS=OUT/'02_mask_arms'; OUTPUTS=OUT/'03_corrected_pages'; LOCKS=OUT/'04_locks'; SCORE=OUT/'05_scoring'; VISUAL=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,OUTPUTS,LOCKS,SCORE,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
ARMS=['reviewed_comparator','raw_yolo','confidence_gated_yolo','reviewer_corrected_yolo']
print('Output:',OUT)

## 2. Integrity helpers

In [ ]:
# PROVENANCE: consolidated project integrity helpers.
def require(condition,message):
    if not bool(condition):
        try:(AUDIT/'fatal_failure.json').write_text(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'message':str(message)},indent=2))
        except Exception:pass
        raise RuntimeError(message)

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def sha256_file(path):
    path=Path(path); require(path.exists() and path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()

def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value):
    path=Path(path);tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)

def read_gray(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE);require(im is not None,f'Unreadable image: {path}');return im

def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)

## 3. Resolve the complete Notebook 06/07 audit intersection

In [ ]:
# PROVENANCE: adapted from Notebook 09 `read_audit_records`, `matches_by_hash` and `required_path`.
# PURPOSE: join Notebook 06 original-stem artefacts to Notebook 07 PageXXX artefacts by source_md5.
def load_audits(folder,audit_type):
    records=[];failures=[]
    for path in sorted(Path(folder).glob('*.json')):
        try:
            record=json.loads(path.read_text())
            if not isinstance(record,dict):raise ValueError('audit root must be an object')
            record['_audit_path']=str(path);record['_audit_type']=audit_type;records.append(record)
        except Exception as error:
            failures.append({'audit_type':audit_type,'audit_path':str(path),'error':str(error)})
    return records,failures

def unique_by_source_md5(records,audit_type):
    result={}
    for record in records:
        source_md5=str(record.get('source_md5','')).strip().lower()
        require(source_md5,f'{audit_type} audit lacks source_md5: {record["_audit_path"]}')
        require(source_md5 not in result,f'Duplicate {audit_type} audits for source_md5 {source_md5}: {result[source_md5]["_audit_path"]} and {record["_audit_path"]}')
        result[source_md5]=record
    return result

def required_audit_path(record,field,audit_type):
    raw=record.get(field);require(raw,f'{audit_type} audit missing {field}: {record["_audit_path"]}')
    path=Path(raw);require(path.exists(),f'{audit_type} audit references missing {field}: {path}')
    return path

drafts,dfail=load_audits(DRAFT_AUDIT_DIR,'Notebook 06 draft')
reviews,rfail=load_audits(REVIEW_AUDIT_DIR,'Notebook 07 review')
failures=dfail+rfail
if failures:
    pd.DataFrame(failures).to_csv(AUDIT/'audit_parse_failures.csv',index=False)
    raise RuntimeError(f'{len(failures)} HITL audit records could not be read.')

draft_by_md5=unique_by_source_md5(drafts,'Notebook 06 draft')
review_by_md5=unique_by_source_md5(reviews,'Notebook 07 review')
paired_md5=sorted(set(draft_by_md5)&set(review_by_md5))
require(paired_md5,'No source_md5 values occur in both Notebook 06 draft audits and Notebook 07 review audits.')

resolved=[]
for source_md5 in paired_md5:
    draft=draft_by_md5[source_md5];review=review_by_md5[source_md5]
    page=str(review.get('page','')).strip() or Path(review['_audit_path']).stem
    require(re.fullmatch(r'Page\d+',page,re.IGNORECASE),f'Unexpected Notebook 07 page identity {page!r}: {review["_audit_path"]}')
    source_path=required_audit_path(draft,'source_file','Notebook 06')
    raw_mask_path=required_audit_path(draft,'draft_mask','Notebook 06')
    confidence_path=required_audit_path(draft,'confidence_map','Notebook 06')
    observed_path=required_audit_path(review,'aligned_output','Notebook 07')
    reviewed_mask_path=required_audit_path(review,'corrected_mask_output','Notebook 07')
    require(md5_file(source_path).lower()==source_md5,f'{page}: Notebook 06 source_file does not match source_md5')
    if review.get('source') and Path(review['source']).exists():require(md5_file(Path(review['source'])).lower()==source_md5,f'{page}: Notebook 07 source does not match source_md5')
    observed=read_gray(observed_path);raw=read_gray(raw_mask_path);reviewed=read_gray(reviewed_mask_path);confidence=np.load(confidence_path).astype(np.float32)
    require(observed.shape==raw.shape==reviewed.shape==confidence.shape,f'{page}: shape mismatch observed={observed.shape}, raw={raw.shape}, reviewed={reviewed.shape}, confidence={confidence.shape}')
    require(np.isfinite(confidence).all(),f'{page}: confidence contains non-finite values')
    resolved.append({'page':page,'source_md5':source_md5,'original_source_path':str(source_path),'observed_path':str(observed_path),'raw_yolo_mask_path':str(raw_mask_path),'confidence_map_path':str(confidence_path),'reviewed_mask_path':str(reviewed_mask_path),'draft_audit_path':draft['_audit_path'],'review_audit_path':review['_audit_path'],'draft_model_path':draft.get('model_path'),'draft_model_md5':draft.get('model_md5'),'draft_prediction_confidence':draft.get('prediction_confidence')})

resolution=pd.DataFrame(resolved).sort_values('page').reset_index(drop=True)
require(not resolution.page.duplicated().any(),'Duplicate Notebook 07 Page identities found.')
if PAGE_ALLOWLIST is not None:
    requested=set(PAGE_ALLOWLIST);missing=sorted(requested-set(resolution.page));require(not missing,f'Allowlisted pages lack complete Notebook 06/07 lineage: {missing}')
    resolution=resolution[resolution.page.isin(requested)].copy()
require(len(resolution)>0,'No pages remain after allowlist filtering.')
resolution.to_csv(AUDIT/'hitl_input_resolution.csv',index=False)
print('Eligible paired pages:',len(resolution))
display(resolution[['page','original_source_path','observed_path','raw_yolo_mask_path','confidence_map_path','reviewed_mask_path']])

## 4. Validate the established YOLO gate

In [ ]:
# PROVENANCE: NEW V9.0A control.
# PURPOSE: keep the raw Notebook 06 draft threshold separate from the previously established higher-confidence gate.
require(ESTABLISHED_YOLO_GATE is not None,'Set ESTABLISHED_YOLO_GATE to the exact previously established YOLO confidence-gating threshold.')
require(ESTABLISHED_YOLO_GATE_POLICY_ID is not None and str(ESTABLISHED_YOLO_GATE_POLICY_ID).strip(),'Set ESTABLISHED_YOLO_GATE_POLICY_ID to the prior experiment/policy identity.')
ESTABLISHED_YOLO_GATE=float(ESTABLISHED_YOLO_GATE)
require(0<=ESTABLISHED_YOLO_GATE<=1,'ESTABLISHED_YOLO_GATE must be in [0,1].')
raw_thresholds=pd.to_numeric(resolution.draft_prediction_confidence,errors='coerce').dropna().unique().tolist()
require(len(raw_thresholds)<=1,f'Notebook 06 draft audits disagree on prediction_confidence: {raw_thresholds}')
raw_threshold=float(raw_thresholds[0]) if raw_thresholds else None
require(raw_threshold is None or ESTABLISHED_YOLO_GATE>raw_threshold,'Established confidence gate must be higher than the raw Notebook 06 draft threshold; otherwise the gated and raw arms collapse.')
gate_record={'established_yolo_gate':ESTABLISHED_YOLO_GATE,'policy_id':str(ESTABLISHED_YOLO_GATE_POLICY_ID),'raw_notebook06_prediction_confidence':raw_threshold,'optimised_in_v9_0a':False}
atomic_json(AUDIT/'yolo_gate_identity.json',gate_record)
print(json.dumps(gate_record,indent=2))

## 5. Canonical correction implementation

In [ ]:
# PROVENANCE: TRANSPLANTED from Notebook 09 canonical V7.8 structural block.
# PURPOSE: run the identical observable-only correction implementation for each authoritative seed mask.
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0];require(np.array_equal(out[mask==0],original[mask==0]),'Outside-mask drift');return out

def prepare_mask(seed,dilation=MASK_DILATION):
    mask=binary(seed)
    if dilation>0:mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return mask

def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False;merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop();rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else:rest.append((bx0,by0,bx1,by1))
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def telea_reconstruct(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
    return immutable(observed,result,mask)

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32);num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma);return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def support_maps(observed,mask):
    dark=(observed<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0;preserve=m&(horizontal|vertical|compact)&(observed<200);remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD);ambiguous=m&~preserve&~remove;result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve];return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact}

## 6. Build mask arms and execute the frozen correction chain

In [ ]:
# PROVENANCE: NEW V9.0A orchestration around the canonical correction block.
runtime={};mask_rows=[];output_rows=[];edit_rows=[]
for _,r in resolution.iterrows():
    page=r.page;observed=read_gray(r.observed_path);raw=binary(read_gray(r.raw_yolo_mask_path));reviewed=binary(read_gray(r.reviewed_mask_path));confidence=np.load(r.confidence_map_path).astype(np.float32);gated=np.where((raw>0)&(confidence>=ESTABLISHED_YOLO_GATE),255,0).astype(np.uint8)
    arms={'reviewed_comparator':reviewed,'raw_yolo':raw,'confidence_gated_yolo':gated,'reviewer_corrected_yolo':reviewed.copy()}
    runtime[page]={}
    edit_rows.append({'page':page,'raw_pixels':int((raw>0).sum()),'reviewed_pixels':int((reviewed>0).sum()),'reviewer_additions':int(((reviewed>0)&(raw==0)).sum()),'reviewer_erasures':int(((reviewed==0)&(raw>0)).sum()),'raw_reviewed_xor':int(np.logical_xor(raw>0,reviewed>0).sum())})
    for arm,seed in arms.items():
        d=OUTPUTS/arm/page;d.mkdir(parents=True,exist_ok=False);m=prepare_mask(seed,MASK_DILATION);telea=telea_reconstruct(observed,m);corrected,routes=geometry_t80(observed,m,telea)
        require(np.array_equal(corrected[m==0],observed[m==0]),f'{page}/{arm}: outside-mask drift')
        md=MASKS/arm;md.mkdir(parents=True,exist_ok=True);seed_path=md/f'{page}.png';cv2.imwrite(str(seed_path),seed)
        for name,a in [('observed',observed),('authoritative_seed',seed),('prepared_mask',m),('telea',telea),('corrected',corrected),('route_preserve',routes['preserve'].astype(np.uint8)*255),('route_remove',routes['remove'].astype(np.uint8)*255),('route_ambiguous',routes['ambiguous'].astype(np.uint8)*255)]:cv2.imwrite(str(d/f'{name}.png'),a)
        runtime[page][arm]={'seed':seed,'mask':m,'corrected':corrected,'routes':routes}
        mask_rows.append({'page':page,'arm':arm,'seed_pixels':int((seed>0).sum()),'prepared_mask_pixels':int((m>0).sum()),'seed_sha256':array_hash(seed),'prepared_mask_sha256':array_hash(m),'source_md5':r.source_md5})
        output_rows.append({'page':page,'arm':arm,'corrected_sha256':array_hash(corrected),'outside_mask_drift':0,'changed_pixels':int((corrected!=observed).sum())})
mask_manifest=pd.DataFrame(mask_rows);output_manifest=pd.DataFrame(output_rows);review_edits=pd.DataFrame(edit_rows)
mask_manifest.to_csv(LOCKS/'mask_arm_manifest.csv',index=False);output_manifest.to_csv(LOCKS/'corrected_output_manifest.csv',index=False);review_edits.to_csv(SCORE/'reviewer_edits.csv',index=False)
# Explicitly prove the two reviewed-role arms are identical.
require(all(np.array_equal(runtime[p]['reviewed_comparator']['seed'],runtime[p]['reviewer_corrected_yolo']['seed']) for p in runtime),'Reviewed-role arms unexpectedly differ.')
print('Pages processed:',len(runtime),'arm runs:',len(output_manifest))

## 7. Prediction and output lock

In [ ]:
# PROVENANCE: project truth-firewall pattern; purpose: lock all inference artefacts before loading clean truth.
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'pages':sorted(runtime),'source_md5_roster':resolution[['page','source_md5']].sort_values('page').to_dict('records'),'mask_manifest_sha256':sha256_file(LOCKS/'mask_arm_manifest.csv'),'output_manifest_sha256':sha256_file(LOCKS/'corrected_output_manifest.csv'),'hitl_resolution_sha256':sha256_file(AUDIT/'hitl_input_resolution.csv'),'gate_identity_sha256':sha256_file(AUDIT/'yolo_gate_identity.json'),'correction_contract':{'mask_dilation':MASK_DILATION,'crop_margin':CROP_MARGIN,'geometry_threshold':GEOMETRY_THRESHOLD,'band_width':BAND_WIDTH},'truth_loaded':False}
lock['lock_id']=object_hash(lock);atomic_json(LOCKS/'prediction_and_output_lock.json',lock)
print('Lock ID:',lock['lock_id'])

## 8. Post-lock paired comparisons

In [ ]:
# PROVENANCE: NEW V9.0A paired mask-authority analysis.
def mask_metrics(candidate,reference):
    c=candidate>0;r=reference>0;inter=int((c&r).sum());union=int((c|r).sum());fp=int((c&~r).sum());fn=int((~c&r).sum());return {'iou':inter/max(union,1),'false_positive_pixels_vs_reviewed':fp,'false_negative_pixels_vs_reviewed':fn,'xor_pixels':int((c^r).sum())}
rows=[]
for page,arms in runtime.items():
    rm=arms['reviewed_comparator']['seed'];ro=arms['reviewed_comparator']['corrected']
    for arm,z in arms.items():
        q=mask_metrics(z['seed'],rm);q.update({'page':page,'arm':arm,'corrected_xor_pixels_vs_reviewed':int((z['corrected']!=ro).sum()),'corrected_mae_vs_reviewed':float(np.abs(z['corrected'].astype(np.int16)-ro.astype(np.int16)).mean())});rows.append(q)
paired=pd.DataFrame(rows);paired.to_csv(SCORE/'paired_mask_and_output_evidence.csv',index=False)
summary=paired.groupby('arm',as_index=False).agg(pages=('page','nunique'),mean_mask_iou=('iou','mean'),mask_false_positive_pixels=('false_positive_pixels_vs_reviewed','sum'),mask_false_negative_pixels=('false_negative_pixels_vs_reviewed','sum'),mask_xor_pixels=('xor_pixels','sum'),corrected_xor_pixels=('corrected_xor_pixels_vs_reviewed','sum'),corrected_mae_mean=('corrected_mae_vs_reviewed','mean'))
summary.to_csv(SCORE/'arm_summary.csv',index=False);display(summary)

## 9. Optional clean-truth scoring after lock

In [ ]:
# PROVENANCE: project clean-truth firewall; purpose: score only after prediction/output lock.
require((LOCKS/'prediction_and_output_lock.json').exists(),'Prediction/output lock missing.')
clean_map={p.stem:p for p in CLEAN_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if CLEAN_DIR.exists() else {}
truth_rows=[];missing=[]
for page,arms in runtime.items():
    cp=clean_map.get(page)
    if cp is None:missing.append(page);continue
    clean=read_gray(cp)
    for arm,z in arms.items():
        require(clean.shape==z['corrected'].shape,f'{page}: clean/output shape mismatch')
        diff=np.abs(clean.astype(np.float32)-z['corrected'].astype(np.float32));truth_rows.append({'page':page,'arm':arm,'whole_page_mae':float(diff.mean()),'inside_prepared_mask_mae':float(diff[z['mask']>0].mean()) if np.any(z['mask']>0) else np.nan})
if truth_rows:pd.DataFrame(truth_rows).to_csv(SCORE/'postlock_clean_truth_scores.csv',index=False)
if missing:pd.DataFrame({'page_missing_clean_counterpart':missing}).to_csv(AUDIT/'clean_truth_missing_pages.csv',index=False)
print('Truth-scored pages:',len(set(x['page'] for x in truth_rows)),'missing clean:',len(missing))

## 10. Visual audit queue

In [ ]:
# PROVENANCE: NEW V9.0A visual review package.
queue=paired[paired.arm.isin(['raw_yolo','confidence_gated_yolo'])].copy();queue['priority']=queue.corrected_xor_pixels_vs_reviewed+5*queue.false_positive_pixels_vs_reviewed+3*queue.false_negative_pixels_vs_reviewed;queue=queue.sort_values('priority',ascending=False);queue.to_csv(SCORE/'visual_audit_queue.csv',index=False)
for i,r in queue.head(30).reset_index(drop=True).iterrows():
    page=r.page;arm=r.arm;obs=read_gray(resolution.set_index('page').loc[page,'observed_path']);reviewed=runtime[page]['reviewed_comparator']['seed']>0;cand=runtime[page][arm]['seed']>0;overlay=cv2.cvtColor(obs,cv2.COLOR_GRAY2RGB);overlay[reviewed&cand]=(0,180,0);overlay[reviewed&~cand]=(255,165,0);overlay[cand&~reviewed]=(255,0,0)
    fig,ax=plt.subplots(1,4,figsize=(20,7));ims=[obs,overlay,runtime[page]['reviewed_comparator']['corrected'],runtime[page][arm]['corrected']];titles=['Observed','Mask overlap green / missed orange / overreach red','Reviewed comparator correction',arm+' correction']
    for a,im,title in zip(ax,ims,titles):a.imshow(im,cmap='gray' if im.ndim==2 else None);a.set_title(title);a.axis('off')
    fig.suptitle(f'{page} | {arm}');plt.tight_layout();fig.savefig(VISUAL/f'{i+1:03d}_{page}_{arm}.png',dpi=130,bbox_inches='tight');plt.close(fig)
print('Visual cases:',min(30,len(queue)))

## 11. Fail-closed verdict

In [ ]:
# PROVENANCE: NEW V9.0A verdict; evidence package only, no automatic authority.
checks={'audit_join_by_source_md5':True,'canonical_page_identity_from_notebook07':True,'confidence_map_resolved_from_notebook06_audit':True,'established_gate_explicit':True,'all_pages_have_reviewed_masks':True,'four_arm_roster_complete':len(mask_manifest)==len(resolution)*4,'zero_outside_mask_drift':bool((output_manifest.outside_mask_drift==0).all()),'prediction_output_lock_exists':(LOCKS/'prediction_and_output_lock.json').exists(),'paired_comparisons_written':(SCORE/'paired_mask_and_output_evidence.csv').exists()}
first_failure=next((k for k,v in checks.items() if not v),None);verdict='PASS_V9_0A_EVIDENCE_PACKAGE_READY_FOR_REVIEW' if first_failure is None else 'FAIL_V9_0A_INVALID_RUN'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A-audit-joined-mask-authority-transfer','verdict':verdict,'first_failed_gate':first_failure,'checks':checks,'lock_id':lock['lock_id'],'automatic_mask_authority_authorised':False,'production_output_authority':False,'reviewed_mask_is_comparator_not_perfect_truth':True,'reviewed_comparator_and_reviewer_corrected_yolo_are_role_labels_for_the_same_pixels':True}
atomic_json(OUT/'v9_0a_verdict.json',record);print(json.dumps(record,indent=2))
if first_failure:raise RuntimeError(f'V9.0A failed at {first_failure}')

## 12. Final Upload Package

In [ ]:
# PROVENANCE: standard project Final Upload Package pattern.
required=[OUT/'v9_0a_verdict.json',AUDIT/'hitl_input_resolution.csv',AUDIT/'yolo_gate_identity.json',LOCKS/'mask_arm_manifest.csv',LOCKS/'corrected_output_manifest.csv',LOCKS/'prediction_and_output_lock.json',SCORE/'reviewer_edits.csv',SCORE/'paired_mask_and_output_evidence.csv',SCORE/'arm_summary.csv',SCORE/'visual_audit_queue.csv']
if (SCORE/'postlock_clean_truth_scores.csv').exists():required.append(SCORE/'postlock_clean_truth_scores.csv')
for p in required:require(p.exists(),f'Missing upload artefact: {p}')
(UPLOAD/'README.txt').write_text('V9.0A MASK-AUTHORITY TRANSFER REVIEW PACKAGE\nJoins Notebook 06 and Notebook 07 by source_md5 and uses Notebook 07 PageXXX as canonical identity. Research evidence only. No automatic mask authority is authorised.\n')
rows=[]
for src in required+sorted(VISUAL.glob('*.png'))[:30]:
    rel=('visuals/'+src.name) if src.suffix.lower()=='.png' else ('tables/'+src.name if src.parent==SCORE else 'audit/'+src.name if src.parent in [AUDIT,LOCKS] else src.name);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in rows))
zip_path=UPLOAD/'58_V9_0A_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.');print('Upload bundle:',zip_path)

## Required configuration before execution

Set `ESTABLISHED_YOLO_GATE` and `ESTABLISHED_YOLO_GATE_POLICY_ID` to the exact prior YOLO gating policy. The notebook will not reinterpret Notebook 06 `prediction_confidence` as that higher-confidence gate.

The roster is otherwise derived automatically from the intersection of Notebook 06 and Notebook 07 audit records. No manually copied cohort directory and no filename-stem join are required.